# Speed of Light Model Metric Calculations

Apply model metrics to Speed of Light formulas to get actual values.

In [1]:
# Model metrics.
SCENE_MODEL_INPUTS = {
    "Bicycle": {"p": 4_795_745, "v": 4_795_745, "width": 1_237, "height": 822, "instances": 3_167_644},
    "Bonsai": {"p": 1_059_559, "v": 1_059_559, "width": 780, "height": 520, "instances": 779_696},
    "Counter": {"p": 989_768, "v": 989_768, "width": 779, "height": 519, "instances": 1_134_338},
    "Dr. Johnson": {"p": 3_230_146, "v": 3_230_146, "width": 1_332, "height": 876, "instances": 1_889_651},
    "Flowers": {"p": 2_822_498, "v": 2_822_498, "width": 1_256, "height": 828, "instances": 1_665_163},
    "Garden": {"p": 4_160_377, "v": 4_160_377, "width": 1_297, "height": 840, "instances": 3_102_843},
    "Kitchen": {"p": 1_533_692, "v": 1_533_692, "width": 779, "height": 520, "instances": 1_650_336},
    "Playroom": {"p": 1_899_320, "v": 1_899_320, "width": 1_264, "height": 832, "instances": 1_302_347},
    "Room": {"p": 1_154_398, "v": 1_154_398, "width": 779, "height": 519, "instances": 830_436},
    "Stump": {"p": 4_236_367, "v": 4_236_367, "width": 1_245, "height": 825, "instances": 1_952_736},
    "Train": {"p": 1_061_137, "v": 1_061_137, "width": 980, "height": 545, "instances": 1_740_123},
    "Treehill": {"p": 3_198_653, "v": 3_198_653, "width": 1_267, "height": 832, "instances": 2_014_036},
    "Truck": {"p": 2_010_367, "v": 2_010_367, "width": 979, "height": 546, "instances": 1_718_863},
}

def _compute_model(expression, *, width: int, height: int, instances: int) -> int:
    return int(expression.evalf(subs={
        W: width,
        H: height,
        N: instances,
    }).round())


def _compute_scene_model(expression, scene_name: str) -> int:
    scene = SCENE_MODEL_INPUTS[scene_name]
    return _compute_model(
        expression.subs({P: scene["p"], V: scene["v"]}),
        width=scene["width"],
        height=scene["height"],
        instances=scene["instances"],
    )


def compute_bicycle(expression) -> int:
    return _compute_scene_model(expression, "Bicycle")


def compute_bonsai(expression) -> int:
    return _compute_scene_model(expression, "Bonsai")


def compute_counter(expression) -> int:
    return _compute_scene_model(expression, "Counter")


def compute_drjohnson(expression) -> int:
    return _compute_scene_model(expression, "Dr. Johnson")


def compute_flowers(expression) -> int:
    return _compute_scene_model(expression, "Flowers")


def compute_garden(expression) -> int:
    return _compute_scene_model(expression, "Garden")


def compute_kitchen(expression) -> int:
    return _compute_scene_model(expression, "Kitchen")


def compute_playroom(expression) -> int:
    return _compute_scene_model(expression, "Playroom")


def compute_room(expression) -> int:
    return _compute_scene_model(expression, "Room")


def compute_stump(expression) -> int:
    return _compute_scene_model(expression, "Stump")


def compute_train(expression) -> int:
    return _compute_scene_model(expression, "Train")


def compute_treehill(expression) -> int:
    return _compute_scene_model(expression, "Treehill")


def compute_truck(expression) -> int:
    return _compute_scene_model(expression, "Truck")


In [2]:
# Type sizes (bytes).
UINT = 4
UINT2 = 8
USHORT = 2
USHORT4 = 8
FLOAT = 4
FLOAT2 = 8
FLOAT3 = 12
FLOAT4 = 16

## Device metrics

In [3]:
MAX_BANDWIDTH_BYTES = 936 * 1e9
MAX_FLOPS = 35.6 * 1e12

# Formulas

In [4]:
import csv
from pathlib import Path

from sympy import ceiling, symbols, init_printing, print_latex

init_printing()


def kernel_seconds(kernel_io: float, kernel_flop: float) -> float:
    return kernel_io / MAX_BANDWIDTH_BYTES + kernel_flop / MAX_FLOPS


def kernel_microseconds(kernel_io: float, kernel_flop: float) -> float:
    return round(kernel_seconds(kernel_io, kernel_flop) * 1e6)


def cprint(value):
    print(f"{value:,}")


def _load_scene_flops() -> dict[str, int]:
    metrics_path = next((
        path
        for path in (
            Path("SCEB3DGS paper metrics.csv"),
            Path("src/model_exploration/SCEB3DGS paper metrics.csv"),
        )
        if path.exists()
    ), None)
    if metrics_path is None:
        raise FileNotFoundError("Could not find SCEB3DGS paper metrics.csv")
    with metrics_path.open(newline="") as handle:
        rows = list(csv.reader(handle))
    if len(rows) < 2:
        raise ValueError("SCEB3DGS paper metrics.csv must contain a header row and a flop row")
    return {scene: int(value) for scene, value in zip(rows[0], rows[1])}


SCENE_FLOPS = _load_scene_flops()

MODEL_METRICS = [
    ("Bicycle", compute_bicycle),
    ("Bonsai", compute_bonsai),
    ("Counter", compute_counter),
    ("Dr. Johnson", compute_drjohnson),
    ("Flowers", compute_flowers),
    ("Garden", compute_garden),
    ("Kitchen", compute_kitchen),
    ("Playroom", compute_playroom),
    ("Room", compute_room),
    ("Stump", compute_stump),
    ("Train", compute_train),
    ("Treehill", compute_treehill),
    ("Truck", compute_truck),
]

SCENE_METRICS = [
    (model_name, compute_model, SCENE_FLOPS[model_name])
    for model_name, compute_model in MODEL_METRICS
]


def kernel_metrics(memory_expression, scene_metrics):
    print("Memory expression")
    print_latex(memory_expression)
    print()
    model_rows = []
    for model_name, compute_model, kernel_flop in scene_metrics:
        print(f"{model_name} bytes")
        model_bytes = compute_model(memory_expression)
        cprint(model_bytes)
        print()
        print(f"{model_name} SOL duration (microseconds)")
        sol_duration = kernel_microseconds(model_bytes, kernel_flop)
        cprint(sol_duration)
        print()
        model_rows.append((model_name, model_bytes, kernel_flop, sol_duration))
    return model_rows


def resolve_output_path(filename: str) -> Path:
    base = Path("src/model_exploration") if Path("src/model_exploration").exists() else Path(".")
    return base / filename


def write_kernel_metrics_csv(output_path: Path, rows):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["model name", "bytes", "fp32", "sol duration"])
        writer.writerows(rows)


W, H, P, V, N = symbols("W H P V N", real=True)
T = ceiling(W / 16) * ceiling(H / 16)


In [7]:
print_latex(T)

\left\lceil{\frac{H}{16}}\right\rceil \left\lceil{\frac{W}{16}}\right\rceil


## Blend

In [6]:
blend_io = UINT2 * T + UINT * N + FLOAT2 * N + FLOAT4 * N + FLOAT3 * N + FLOAT3 * 1 + FLOAT * 3 * W * H
blend_metrics = kernel_metrics(blend_io, SCENE_METRICS)
write_kernel_metrics_csv(resolve_output_path("sol_blend.csv"), blend_metrics)


Memory expression
12 H W + 40 N + 8 \left\lceil{\frac{H}{16}}\right\rceil \left\lceil{\frac{W}{16}}\right\rceil + 12

Bicycle bytes
138,939,988

Bicycle SOL duration (microseconds)
404

Bonsai bytes
36,067,988

Bonsai SOL duration (microseconds)
102

Counter bytes
50,238,080

Counter SOL duration (microseconds)
137

Dr. Johnson bytes
89,624,996

Dr. Johnson SOL duration (microseconds)
273

Flowers bytes
79,119,012

Flowers SOL duration (microseconds)
212

Garden bytes
137,222,260

Garden SOL duration (microseconds)
407

Kitchen bytes
70,887,348

Kitchen SOL duration (microseconds)
214

Playroom bytes
64,746,532

Playroom SOL duration (microseconds)
207

Room bytes
38,082,000

Room SOL duration (microseconds)
115

Stump bytes
90,467,400

Stump SOL duration (microseconds)
263

Train bytes
76,031,492

Train SOL duration (microseconds)
165

Treehill bytes
93,244,460

Treehill SOL duration (microseconds)
257

Truck bytes
75,186,300

Truck SOL duration (microseconds)
194

